# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan **Polars LazyFrame** dan **DuckDB**.

**Dataset:** cuaca per jam untuk 40 kota di 38 provinsi Indonesia, 2001-2024, bersumber dari **NASA POWER** (reanalisis MERRA-2). Detail sumber, lisensi, dan cara mengunduh ada di `data/README.md`.

Notebook ini hanya membaca rencana query (`scan_parquet`) sampai `collect()` dipanggil, sehingga file besar tidak dimuat seluruhnya ke memori.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'cuaca_indonesia.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Jalankan `python src/download_data.py` untuk mengunduhnya ke data/raw/.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: cuaca_indonesia.parquet
Ukuran file: 190.2 MB


## 1. Profil Skema dan Ringkasan dengan Polars

`scan_parquet` membangun rencana query secara *lazy*. Skema dan ringkasan dihitung tanpa memuat seluruh kolom ke memori.

In [2]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

Schema({'waktu_lokal': Datetime(time_unit='us', time_zone=None), 'T2M': Float64, 'T2MDEW': Float64, 'RH2M': Float64, 'QV2M': Float64, 'PRECTOTCORR': Float64, 'PS': Float64, 'WS10M': Float64, 'WD10M': Float64, 'ALLSKY_SFC_SW_DWN': Float64, 'CLOUD_AMT': Float64, 'kota': String, 'provinsi': String, 'latitude': Float64, 'longitude': Float64, 'zona_waktu': String, 'utc_offset_jam': Int8})


In [3]:
ringkasan = lazy_data.select(
    pl.len().alias('jumlah_baris'),
    pl.col('kota').n_unique().alias('jumlah_kota'),
    pl.col('provinsi').n_unique().alias('jumlah_provinsi'),
    pl.col('waktu_lokal').min().alias('waktu_min'),
    pl.col('waktu_lokal').max().alias('waktu_max'),
).collect()
ringkasan

jumlah_baris,jumlah_kota,jumlah_provinsi,waktu_min,waktu_max
u32,u32,u32,datetime[μs],datetime[μs]
8415360,40,38,2001-01-01 00:00:00,2024-12-31 23:00:00


In [4]:
jumlah_baris = ringkasan['jumlah_baris'][0]
null_summary = (
    lazy_data.select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(4).alias('persen_null')
    )
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""waktu_lokal""",0,0.0
"""T2M""",0,0.0
"""T2MDEW""",0,0.0
"""RH2M""",0,0.0
"""QV2M""",0,0.0
…,…,…
"""provinsi""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0


In [5]:
contoh_data = lazy_data.head(10).collect()
contoh_data

waktu_lokal,T2M,T2MDEW,RH2M,QV2M,PRECTOTCORR,PS,WS10M,WD10M,ALLSKY_SFC_SW_DWN,CLOUD_AMT,kota,provinsi,latitude,longitude,zona_waktu,utc_offset_jam
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,i8
2001-01-01 00:00:00,26.78,23.61,82.66,18.34,0.0,99.85,1.05,141.6,0.0,58.66,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 01:00:00,26.76,23.51,82.27,18.24,0.0,99.79,0.94,138.0,0.0,42.09,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 02:00:00,26.75,23.42,81.88,18.15,0.0,99.75,0.84,135.5,0.0,23.92,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 03:00:00,26.74,23.37,81.66,18.09,0.0,99.74,0.81,129.0,0.0,40.95,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 04:00:00,26.74,23.36,81.68,18.09,0.0,99.78,0.98,112.9,0.0,34.17,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 05:00:00,26.69,23.46,82.34,18.17,0.0,99.84,1.32,99.6,0.0,61.07,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 06:00:00,26.59,23.67,83.91,18.4,0.0,99.91,1.76,92.6,13.82,61.2,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 07:00:00,26.83,24.23,85.55,19.02,0.0,99.98,1.88,88.2,150.38,58.65,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9
2001-01-01 08:00:00,27.33,24.07,82.29,18.83,0.0,100.02,1.74,86.7,345.98,58.77,"""Ambon""","""Maluku""",-3.6954,128.1814,"""WIT""",9


## 2. Profil Statistik dengan DuckDB `SUMMARIZE`

DuckDB membaca Parquet langsung dari disk dan menghitung statistik tiap kolom (tipe, null%, approx. distinct, min, max, rata-rata, kuartil) dalam satu perintah SQL.

In [6]:
connection = duckdb.connect()
profil_duckdb = pl.from_arrow(
    connection.sql(
        'SUMMARIZE SELECT * FROM read_parquet(?)', params=[str(DATA_PATH)]
    ).to_arrow_table()
)
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""waktu_lokal""","""TIMESTAMP""","""2001-01-01 00:00:00""","""2024-12-31 23:00:00""",204930,"""2012-12-31 23:30:00""",null,"""2007-01-08 00:49:42.958797""","""2012-12-26 06:15:45.564162""","""2018-12-26 15:51:53.125955""",8415360,0.00
"""T2M""","""DOUBLE""","""2.85""","""39.45""",3170,"""25.79794205833153""","""3.1742622384504333""","""24.46483505705083""","""26.147541952413153""","""27.697510728199163""",8415360,0.00
"""T2MDEW""","""DOUBLE""","""-6.48""","""28.47""",2606,"""22.74126197215558""","""2.528083121538933""","""22.066785132937078""","""23.412446409664746""","""24.226107957820833""",8415360,0.00
"""RH2M""","""DOUBLE""","""15.21""","""100.0""",7722,"""84.13281654854968""","""11.094820115886813""","""77.92625781317624""","""86.03840235026915""","""92.53661428587682""",8415360,0.00
"""QV2M""","""DOUBLE""","""3.12""","""24.95""",1812,"""17.807909977707435""","""1.8954380567346099""","""17.050082375478095""","""18.20042784458975""","""19.02272629846178""",8415360,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""provinsi""","""VARCHAR""","""Aceh""","""Sumatera Utara""",39,null,null,null,null,null,8415360,0.00
"""latitude""","""DOUBLE""","""-10.1772""","""5.5483""",41,"""-3.0325624999998935""","""3.67486482895683""","""-6.148353111406152""","""-2.8644507998495317""","""-0.6491863986281983""",8415360,0.00
"""longitude""","""DOUBLE""","""95.3238""","""140.669""",40,"""115.96157499998009""","""11.83368463534927""","""106.13170927157313""","""114.99114100710463""","""123.31234748584569""",8415360,0.00


## 3. Ringkasan per Wilayah

Agregasi ringan per kota/provinsi untuk memastikan cakupan spasial lengkap dan memberi gambaran awal suhu serta curah hujan.

In [7]:
ringkasan_kota = (
    lazy_data.group_by('provinsi', 'kota', 'zona_waktu')
    .agg(
        pl.len().alias('jumlah_baris'),
        pl.col('T2M').mean().round(2).alias('suhu_rata2_c'),
        pl.col('T2M').min().round(2).alias('suhu_min_c'),
        pl.col('T2M').max().round(2).alias('suhu_max_c'),
        pl.col('PRECTOTCORR').sum().round(1).alias('total_hujan_mm'),
    )
    .sort('provinsi')
    .collect()
)
ringkasan_kota

provinsi,kota,zona_waktu,jumlah_baris,suhu_rata2_c,suhu_min_c,suhu_max_c,total_hujan_mm
str,str,str,u32,f64,f64,f64,f64
"""Aceh""","""Banda Aceh""","""WIB""",210384,26.88,22.37,33.13,48691.6
"""Bali""","""Denpasar""","""WITA""",210384,25.14,18.69,34.49,39622.4
"""Banten""","""Serang""","""WIB""",210384,27.27,22.12,34.38,51619.1
"""Bengkulu""","""Bengkulu""","""WIB""",210384,25.86,20.3,33.42,76619.9
"""DI Yogyakarta""","""Yogyakarta""","""WIB""",210384,25.88,18.52,35.02,53205.2
…,…,…,…,…,…,…,…
"""Sulawesi Tenggara""","""Kendari""","""WITA""",210384,26.58,20.35,36.36,52379.4
"""Sulawesi Utara""","""Manado""","""WITA""",210384,26.73,23.32,31.89,53124.9
"""Sumatera Barat""","""Padang""","""WIB""",210384,23.83,18.57,29.71,87471.8


In [8]:
connection.close()

## Catatan Milestone 1

**Bentuk data.** Dataset berisi **8.415.360 baris x 17 kolom** (190,2 MB Parquet): 40 kota di 38 provinsi, periode 1 Januari 2001 - 31 Desember 2024 (8.766 hari x 24 jam x 40 kota). Tipe kolom: 1 datetime, 13 numerik, dan 3 string.

**Kualitas dan missing values.** Tidak ada nilai null pada seluruh kolom (0,00%) karena *fill value* `-999` dari NASA POWER sudah dinormalisasi menjadi `null` saat pengunduhan. Dengan demikian penanganan pada Milestone 2 difokuskan pada **outlier dan validasi konsistensi**, bukan imputasi nilai hilang.

**Rentang nilai utama.**

| Kolom | Min | Maks | Rata-rata | Std |
|---|---|---|---|---|
| `T2M` (degC) | 2,85 | 39,45 | 25,80 | 3,17 |
| `T2MDEW` (degC) | -6,48 | 28,47 | 22,74 | 2,53 |
| `RH2M` (%) | 15,21 | 100,00 | 84,13 | 11,10 |
| `QV2M` (g/kg) | 3,12 | 24,95 | 17,81 | 1,90 |
| `PRECTOTCORR` (mm/jam) | 0,00 | 65,92 | 0,27 | 0,59 |
| `PS` (kPa) | 73,82 | 101,67 | 98,26 | 4,54 |
| `WS10M` (m/s) | 0,00 | 19,44 | 2,42 | 1,55 |
| `ALLSKY_SFC_SW_DWN` (W/m2) | 0,00 | 1.073,97 | 208,49 | 281,46 |
| `CLOUD_AMT` (%) | 0,00 | 100,00 | 76,92 | 29,44 |

**Pola awal.** Kota dengan rata-rata suhu tertinggi adalah Tanjung Pinang (27,91 degC) dan terendah Wamena (13,43 degC) - konsisten dengan perbedaan ketinggian. Total curah hujan 2001-2024 tertinggi di Nabire, Padang, dan Pontianak; terendah di Kupang, Mataram, dan Denpasar.

**Anomali yang perlu ditangani pada Milestone 2.**

1. Distribusi `PRECTOTCORR` dan `WS10M` sangat miring ke kanan (hujan ekstrem sampai 65,92 mm/jam, angin sampai 19,44 m/s) sehingga deteksi outlier dengan IQR atau z-score perlu dilakukan.
2. Validasi konsistensi fisik: `T2MDEW <= T2M`, `0 <= RH2M <= 100`, dan `0 <= WD10M <= 360`.
3. Nilai `T2M` minimum 2,85 degC dan `T2MDEW` -6,48 degC berasal dari wilayah dataran tinggi (Wamena) sehingga valid secara fisik dan tidak dibuang tanpa analisis.
4. Perlu diverifikasi tidak ada duplikat pada kombinasi `(kota, waktu_lokal)` dan tidak ada lompatan waktu pada tiap kota.

**Catatan penting.** Timestamp tersimpan dalam **Local Standard Time (LST)** masing-masing kota (bukan UTC), sehingga langsung dapat dipakai untuk analisis siklus harian.